# Era Wallet A-roll proxy builder
Free Google Colab workflow for the large A-roll file already copied into My Drive. It creates a connector-safe MP4 around 205 MiB without downloading the 4.3 GB source to ChatGPT.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, subprocess, hashlib
SOURCE = '/content/drive/MyDrive/ERA_A_ROLL_AUTH_WORKING.mov'
OUTPUT = '/content/drive/MyDrive/ERA_A_ROLL_PROXY_205M.mp4'
SHA_FILE = '/content/drive/MyDrive/ERA_A_ROLL_PROXY_205M.sha256.txt'
TARGET_MIB = 205
assert os.path.exists(SOURCE), f'Missing source: {SOURCE}'
def sh(cmd): return subprocess.check_output(cmd, text=True).strip()
duration = float(sh(['ffprobe','-v','error','-show_entries','format=duration','-of','default=nw=1:nk=1', SOURCE]))
target_bytes = TARGET_MIB * 1024 * 1024
audio_kbps = 88
video_kbps = max(300, min(2600, int((target_bytes*8/duration/1000) - audio_kbps - 24)))
print(f'Duration: {duration:.1f}s | video target: {video_kbps} kbps')
cmd = ['ffmpeg','-hide_banner','-y','-i',SOURCE,'-map','0:v:0','-map','0:a?','-vf',"scale='if(gte(iw,ih),min(1920,iw),-2)':'if(gte(iw,ih),-2,min(1920,ih))',fps=30",'-c:v','libx264','-preset','veryfast','-b:v',f'{video_kbps}k','-maxrate',f'{int(video_kbps*1.2)}k','-bufsize',f'{video_kbps*2}k','-pix_fmt','yuv420p','-c:a','aac','-b:a',f'{audio_kbps}k','-movflags','+faststart',OUTPUT]
subprocess.run(cmd, check=True)
size = os.path.getsize(OUTPUT)
assert size < 235_000_000, f'Output too large: {size} bytes'
h = hashlib.sha256()
with open(OUTPUT,'rb') as f:
    for chunk in iter(lambda: f.read(8*1024*1024), b''): h.update(chunk)
with open(SHA_FILE,'w') as f: f.write(h.hexdigest()+'  '+os.path.basename(OUTPUT)+'\n')
print(f'Created {OUTPUT} ({size/1024/1024:.1f} MiB)')
print('SHA256:', h.hexdigest())


When the last cell prints **Created**, the proxy is in My Drive as `ERA_A_ROLL_PROXY_205M.mp4`. You can close Colab. No paid runtime is required.
